In [ ]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

In [ ]:
nltk.download('stopwords')

In [ ]:
run_on_colab = True

In [ ]:
if run_on_colab:
    jailbreak_df = pd.read_csv('/content/df_jailbreak_categ.csv') #path to jailbreak DF
    regular_df = pd.read_csv('/content/regular_prompts_DAN.csv') #path to regular prompts DF
else:

    jailbreak_df = pd.read_csv('../data/df_jailbreak_categ.csv') #path to jailbreak DF
    regular_df = pd.read_csv('../data/regular_prompts_DAN.csv') #path to regular prompts DF

In [ ]:
jailbreak_df.head(10)

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess_text(text):
    words = text.split()
    processed_words = [stemmer.stem(word) for word in words if word.lower() not in stop_words]
    return ' '.join(processed_words)

In [ ]:
def createTrainingDF(df,regular_df, label_name):
    filtered_df = df[~df['Labels'].apply(lambda labels: label_name in labels)].copy()
    filtered_df['Jailbreak'] = 1
    filtered_df = filtered_df[['prompt', 'Jailbreak']]
    sampled_df = regular_df.sample(n=len(filtered_df), random_state=1).reset_index(drop=True)
    sampled_df['Jailbreak'] = 0
    sampled_df = sampled_df[['prompt', 'Jailbreak']]
    sampled_df['prompt'] = sampled_df['prompt'].apply(preprocess_text)
    merged_df = pd.concat([filtered_df, sampled_df], ignore_index=True)
    merged_df = merged_df.sample(frac=1, random_state=1).reset_index(drop=True)
    return merged_df

In [ ]:
def createTestingDF(df, regular_df, label_name):
    filtered_df = df[df['Labels'].apply(lambda labels: label_name in labels)].copy()
    filtered_df['Jailbreak'] = 1
    filtered_df = filtered_df[['prompt', 'Jailbreak']]
    sampled_df = regular_df.sample(n=len(filtered_df), random_state=1).reset_index(drop=True)
    sampled_df['Jailbreak'] = 0
    sampled_df = sampled_df[['prompt', 'Jailbreak']]
    sampled_df['prompt'] = sampled_df['prompt'].apply(preprocess_text)
    merged_df = pd.concat([filtered_df, sampled_df], ignore_index=True)
    merged_df = merged_df.sample(frac=1, random_state=1).reset_index(drop=True)
    return merged_df

In [ ]:
df_cr_test = createTestingDF(jailbreak_df,regular_df,'Character Roleplay')

In [ ]:
df_cr_train = createTrainingDF(jailbreak_df,regular_df,'Character Roleplay')
df_sup_train = createTrainingDF(jailbreak_df,regular_df,'Superior Model')
df_sudo_train = createTrainingDF(jailbreak_df,regular_df,'Sudo Mode')
df_sim_train = createTrainingDF(jailbreak_df,regular_df,'Simulate Jailbreaking')
df_eth_train = createTrainingDF(jailbreak_df,regular_df,'Ethical Appeal')
df_cr_test = createTestingDF(jailbreak_df,regular_df,'Character Roleplay')
df_sup_test = createTestingDF(jailbreak_df,regular_df,'Superior Model')
df_sudo_test = createTestingDF(jailbreak_df,regular_df,'Sudo Mode')
df_sim_test = createTestingDF(jailbreak_df,regular_df,'Simulate Jailbreaking')
df_eth_test = createTestingDF(jailbreak_df,regular_df,'Ethical Appeal')

In [ ]:
train_df_list = ['df_cr_train' , 'df_sup_train' , 'df_sudo_train' , 'df_sim_train' , 'df_eth_train']
test_df_list = ['df_cr_test' , 'df_sup_test' , 'df_sudo_test' , 'df_sim_test' , 'df_eth_test']

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
class PromptDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer, max_length=128):
        self.prompts = prompts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.prompts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    return {
        'accuracy': acc,
        'f1': f1,
        'tpr': tpr,
        'fnr': fnr
    }

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
metrics_dict = {}

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import os

for i, (train_name, test_name) in enumerate(zip(train_df_list, test_df_list)):
    train_df = globals()[train_name]
    test_df = globals()[test_name]

    train_dataset = PromptDataset(
        prompts=train_df['prompt'].tolist(),
        labels=train_df['Jailbreak'].tolist(),
        tokenizer=tokenizer
    )

    test_dataset = PromptDataset(
        prompts=test_df['prompt'].tolist(),
        labels=test_df['Jailbreak'].tolist(),
        tokenizer=tokenizer
    )

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

    model_output_dir = f'./saved_model_{i}'
    os.makedirs(model_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=10,
        weight_decay=0.01,
        logging_dir=os.path.join(model_output_dir, 'logs'),
        logging_steps=10,
        eval_strategy="epoch"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)
    eval_results = trainer.evaluate()
    metrics_dict[f'Model_{i+1}'] = {
        'accuracy': eval_results.get('eval_accuracy'),
        'f1_score': eval_results.get('eval_f1'),
        'tpr': eval_results.get('eval_tpr'),
        'fnr': eval_results.get('eval_fnr')
    }

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import torch
import os

In [ ]:
training_data_binary = pd.read_csv('/content/training_data_binary.csv')
testing_data_binary = pd.read_csv('/content/testing_data_binary.csv')

In [ ]:
combined_df = pd.concat([training_data_binary, testing_data_binary], ignore_index=True)

In [ ]:
class PromptDataset(torch.utils.data.Dataset):
    def __init__(self, prompts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(prompts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
test_dataset = PromptDataset(
    prompts=combined_df['prompt'].tolist(),
    labels=combined_df['jailbreak'].tolist(),
    tokenizer=tokenizer
)

In [ ]:
def compute_metrics_from_preds(preds, labels):
    preds_class = (preds >= 0.5).astype(int)
    acc = accuracy_score(labels, preds_class)
    auc = roc_auc_score(labels, preds)
    tn, fp, fn, tp = confusion_matrix(labels, preds_class).ravel()
    tpr = tp / (tp + fn)
    fnr = fn / (tp + fn)
    return {
        'accuracy': acc,
        'auc': auc,
        'tpr': tpr,
        'fnr': fnr
    }

In [ ]:
model_results = {}

In [ ]:
for i in range(len(train_df_list)):
    model_path = f'./saved_model_{i}'
    model = BertForSequenceClassification.from_pretrained(model_path)
    tokenizer = BertTokenizer.from_pretrained(model_path)

    trainer = Trainer(model=model)
    preds_output = trainer.predict(test_dataset)
    logits = preds_output.predictions
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    labels = preds_output.label_ids

    metrics = compute_metrics_from_preds(probs, labels)
    model_results[f'Model_{i+1}'] = metrics

In [ ]:
for model_name, metric_dict in model_results.items():
    print(f"{model_name}:")
    for metric, value in metric_dict.items():
        print(f"  {metric}: {value:.4f}")
    print()